# QFT-Graph: Training on Google Colab

End-to-end training pipeline for the heterogeneous GNN on scalar φ⁴ theory.

**Steps:**
1. Mount Google Drive & install dependencies
2. Generate Monte Carlo training data
3. Build heterogeneous graph dataset
4. Train the HeteroGNN model
5. Evaluate energy prediction quality
6. Run coupling sweep for critical exponents

## 0. Setup: Mount Google Drive & Install Dependencies

In [ ]:
import os
import sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Path to the project on your Google Drive
    PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'

    # Install dependencies (Colab already has PyTorch)
    !pip install -q torch-geometric omegaconf

    # Add project source to Python path
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

    # Set working directory so relative paths (data/, experiments/) resolve inside the project
    os.chdir(PROJECT_ROOT)
    print(f'Working directory: {os.getcwd()}')
else:
    # Local: just add src to path
    sys.path.insert(0, '../src')

print('Setup complete.')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from qft_graph.config import (
    LatticeConfig, ScalarFieldConfig, MCConfig, ModelConfig, TrainingConfig
)
from qft_graph.lattice.hypercubic import HypercubicLattice
from qft_graph.fields.scalar import ScalarField
from qft_graph.actions.phi4 import Phi4Action
from qft_graph.mc.metropolis import MetropolisSampler
from qft_graph.mc.observables import ObservableSet
from qft_graph.mc.analysis import jackknife_mean_error
from qft_graph.graphs.builder import HeteroGraphBuilder
from qft_graph.models.hetero_gnn import HeteroGNN
from qft_graph.training.trainer import Trainer
from qft_graph.training.metrics import energy_correlation, relative_error
from qft_graph.utils.reproducibility import set_seed

set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

plt.style.use('dark_background')
%matplotlib inline

## 1. Configuration

All parameters in one place for easy experimentation.

In [ ]:
# === LATTICE ===
LATTICE_SIZE = (16, 16)       # 2D square lattice
LATTICE_SPACING = 1.0

# === PHYSICS ===
MASS_SQUARED = -0.5           # Bare mass squared (near critical)
COUPLING = 0.5                # Quartic coupling lambda

# === MONTE CARLO ===
N_CONFIGS = 5000              # Training configurations
N_THERMALIZATION = 1000       # Thermalization sweeps
N_SWEEPS_BETWEEN = 10         # Decorrelation sweeps
MC_STEP_SIZE = 1.0            # Metropolis proposal width

# === MODEL ===
HIDDEN_DIM = 64               # GNN hidden dimension
N_MP_BLOCKS = 3               # Number of 3-stage message passing blocks
ENCODER_LAYERS = 2            # MLP depth in encoders

# === TRAINING ===
EPOCHS = 150
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
LOSS_FN = 'energy_matching'   # 'energy_matching' | 'kl_divergence' | 'relative_energy'

print(f'Lattice: {LATTICE_SIZE[0]}x{LATTICE_SIZE[1]}, a={LATTICE_SPACING}')
print(f'Physics: m²={MASS_SQUARED}, λ={COUPLING}')
print(f'MC: {N_CONFIGS} configs, {N_THERMALIZATION} therm, {N_SWEEPS_BETWEEN} decorr')
print(f'Model: H={HIDDEN_DIM}, {N_MP_BLOCKS} MP blocks')
print(f'Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}')

## 2. Generate Monte Carlo Data

The Metropolis-Hastings sampler generates field configurations distributed as exp(-S_E[φ]).

In [ ]:
lattice_config = LatticeConfig(dimensions=LATTICE_SIZE, spacing=LATTICE_SPACING)
lattice = HypercubicLattice(lattice_config)
field_config = ScalarFieldConfig(mass_squared=MASS_SQUARED, coupling=COUPLING)
action = Phi4Action(lattice, field_config)

mc_config = MCConfig(
    n_configs=N_CONFIGS,
    n_thermalization=N_THERMALIZATION,
    n_sweeps_between=N_SWEEPS_BETWEEN,
    step_size=MC_STEP_SIZE,
    seed=42,
)
sampler = MetropolisSampler(action, mc_config)

print('Generating Monte Carlo configurations...')
mc_result = sampler.generate(N_CONFIGS)

print(f'\nAcceptance rate: {mc_result.acceptance_rate:.3f}')
print(f'Configurations shape: {mc_result.configurations.shape}')
print(f'Actions - mean: {mc_result.actions.mean():.2f}, std: {mc_result.actions.std():.2f}')

In [ ]:
# Quick sanity checks
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Action distribution
axes[0].hist(mc_result.actions.numpy(), bins=50, alpha=0.7, color='#4488ff')
axes[0].set_xlabel(r'$S_E[\phi]$')
axes[0].set_ylabel('Count')
axes[0].set_title('Action Distribution')

# Field value distribution
axes[1].hist(mc_result.configurations.flatten().numpy(), bins=100, alpha=0.7, color='#44bb88', density=True)
axes[1].set_xlabel(r'$\phi$')
axes[1].set_ylabel('Density')
axes[1].set_title('Field Value Distribution')

# Sample configuration heatmap
axes[2].imshow(mc_result.configurations[0].reshape(LATTICE_SIZE).numpy(),
               cmap='RdBu_r', origin='lower')
axes[2].set_title('Sample Configuration')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')

plt.tight_layout()
plt.show()

## 3. Build Heterogeneous Graph Dataset

Convert MC configurations into PyG HeteroData objects with the bipartite spacetime/field structure.

In [ ]:
scalar_field = ScalarField()
builder = HeteroGraphBuilder(lattice, [scalar_field])

print('Building graph dataset...')
dataset = builder.build_dataset(
    configurations={'scalar': mc_result.configurations},
    actions=mc_result.actions,
)
print(f'Dataset size: {len(dataset)} graphs')

# Inspect one graph
sample = dataset[0]
print(f'\nSample graph:')
print(f'  Node types: {sample.node_types}')
print(f'  Edge types: {sample.edge_types}')
print(f'  Spacetime features: {sample["spacetime"].x.shape}')
print(f'  Scalar features: {sample["scalar"].x.shape}')
print(f'  Target action: {sample.y.item():.4f}')

In [ ]:
# Train/val split
n_train = int(0.8 * len(dataset))
train_dataset = dataset[:n_train]
val_dataset = dataset[n_train:]
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

## 4. Create and Train the HeteroGNN

In [ ]:
model_config = ModelConfig(
    hidden_dim=HIDDEN_DIM,
    n_mp_blocks=N_MP_BLOCKS,
    encoder_layers=ENCODER_LAYERS,
    dropout=0.0,
    activation='gelu',
    readout='energy',
)

model = HeteroGNN(
    config=model_config,
    lattice_dim=lattice.dimension(),
    field_types={'scalar': scalar_field.dof_per_site()},
    lattice_spacing=lattice.lattice_spacing(),
)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'\nArchitecture:')
print(f'  Encoders: spacetime ({lattice.dimension()+1} -> {HIDDEN_DIM}), scalar (1 -> {HIDDEN_DIM}), edge ({lattice.dimension()} -> {HIDDEN_DIM})')
print(f'  Message Passing: {N_MP_BLOCKS} x ThreeStageBlock (field->ST->ST->field)')
print(f'  Readout: EnergyHead (per-site MLP summed over lattice)')

In [ ]:
training_config = TrainingConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=1e-5,
    scheduler='cosine',
    loss=LOSS_FN,
    checkpoint_every=50,
    seed=42,
)

experiment_dir = Path('experiments/runs/colab_run')

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    config=training_config,
    experiment_dir=experiment_dir,
    device=device,
)

print(f'Training on {device} for {EPOCHS} epochs...')
history = trainer.train()

print(f'\nFinal train loss: {history["train_loss"][-1]:.6f}')
print(f'Final val loss: {history["val_loss"][-1]:.6f}')
print(f'Final val correlation: {history["val_corr"][-1]:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].semilogy(epochs, history['train_loss'], label='Train', alpha=0.8)
axes[0].semilogy(epochs, history['val_loss'], label='Val', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['val_corr'], color='#44bb88')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Pearson Correlation')
axes[1].set_title('Energy Prediction Correlation')
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Training: {LATTICE_SIZE[0]}x{LATTICE_SIZE[1]}, m²={MASS_SQUARED}, λ={COUPLING}', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Evaluate Energy Predictions

In [ ]:
model.eval()
all_pred = []
all_true = []

with torch.no_grad():
    for data in val_dataset:
        data = data.to(device)
        output = model(data)
        all_pred.append(output['energy'].cpu())
        all_true.append(data.y.cpu())

pred = torch.cat(all_pred)
true = torch.cat(all_true)

corr = energy_correlation(pred, true)
rel_err = relative_error(pred, true)

print(f'Validation Results:')
print(f'  Pearson correlation: {corr:.4f}')
print(f'  Mean relative error: {rel_err:.4f}')
print(f'  Pred range: [{pred.min():.1f}, {pred.max():.1f}]')
print(f'  True range: [{true.min():.1f}, {true.max():.1f}]')

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(true.numpy(), pred.numpy(), alpha=0.3, s=10, color='#4488ff')
lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel(r'True $S_E[\phi]$')
ax.set_ylabel(r'Predicted $S_E[\phi]$')
ax.set_title(f'Energy Prediction (r = {corr:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 6. Coupling Sweep for Critical Exponents

Scan m² at fixed λ to locate the phase transition and extract ν.
Run at multiple lattice sizes for finite-size scaling.

In [ ]:
# This cell is computationally expensive. Reduce n_configs or m2_steps for faster runs.
SWEEP_CONFIGS = 1000
M2_STEPS = 15
SWEEP_SIZES = [(8, 8), (16, 16)]  # Add (32, 32) for publication quality

m2_values = np.linspace(-1.0, 0.0, M2_STEPS)
sweep_results = {}

for dims in SWEEP_SIZES:
    print(f'\n=== Sweeping L={dims[0]} ===')
    lat = HypercubicLattice(LatticeConfig(dimensions=dims))
    obs = ObservableSet(lat)
    
    mags, chis, xis = [], [], []
    
    for i, m2 in enumerate(m2_values):
        act = Phi4Action(lat, ScalarFieldConfig(mass_squared=m2, coupling=COUPLING))
        samp = MetropolisSampler(act, MCConfig(
            n_configs=SWEEP_CONFIGS, n_thermalization=500,
            n_sweeps_between=5, seed=42+i))
        res = samp.generate(SWEEP_CONFIGS)
        
        m_samples = torch.tensor([obs.abs_magnetization(res.configurations[j]) for j in range(SWEEP_CONFIGS)])
        susc_terms = torch.tensor([obs.susceptibility_term(res.configurations[j]) for j in range(SWEEP_CONFIGS)])
        
        mag_mean, _ = jackknife_mean_error(m_samples)
        chi = lat.num_sites() * (susc_terms.mean().item() - m_samples.mean().item()**2)
        
        xi_list = []
        for j in range(min(50, SWEEP_CONFIGS)):
            G_r = obs.two_point_function(res.configurations[j])
            xi = ObservableSet.correlation_length(G_r)
            if 0 < xi < 1000:
                xi_list.append(xi)
        xi_mean = np.mean(xi_list) if xi_list else 0.0
        
        mags.append(mag_mean)
        chis.append(chi)
        xis.append(xi_mean / dims[0])
        
        if (i+1) % 5 == 0:
            print(f'  m²={m2:.2f}: |M|={mag_mean:.3f}, χ={chi:.1f}, ξ/L={xi_mean/dims[0]:.3f}')
    
    sweep_results[dims[0]] = {
        'mags': mags, 'chis': chis, 'xi_over_L': xis
    }

print('\nSweep complete!')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'8': '#ff6644', '16': '#4488ff', '32': '#44bb88'}

for L, data in sweep_results.items():
    c = colors.get(str(L), '#ffffff')
    axes[0].plot(m2_values, data['mags'], 'o-', color=c, label=f'L={L}', markersize=4)
    axes[1].plot(m2_values, data['chis'], 's-', color=c, label=f'L={L}', markersize=4)
    axes[2].plot(m2_values, data['xi_over_L'], '^-', color=c, label=f'L={L}', markersize=4)

axes[0].set_xlabel(r'$m^2$')
axes[0].set_ylabel(r'$|\langle\phi\rangle|$')
axes[0].set_title('Order Parameter')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel(r'$m^2$')
axes[1].set_ylabel(r'$\chi$')
axes[1].set_title('Susceptibility')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel(r'$m^2$')
axes[2].set_ylabel(r'$\xi / L$')
axes[2].set_title(r'$\xi/L$ Crossing (should cross at $m^2_c$)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(rf'Finite-Size Scaling: $\lambda={COUPLING}$', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Save Results

Save the trained model and sweep data. When running on Colab with Google Drive mounted, everything is saved directly to your Drive — no download step needed.

In [ ]:
# Save model checkpoint
save_dir = Path('experiments/runs/colab_run')
save_dir.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'history': history,
    'config': {
        'lattice_size': LATTICE_SIZE,
        'mass_squared': MASS_SQUARED,
        'coupling': COUPLING,
        'hidden_dim': HIDDEN_DIM,
        'n_mp_blocks': N_MP_BLOCKS,
    },
}, save_dir / 'model_final.pt')

# Save sweep results
import json
sweep_save = {}
for L, data in sweep_results.items():
    sweep_save[str(L)] = {
        'm2_values': m2_values.tolist(),
        'mags': data['mags'],
        'chis': data['chis'],
        'xi_over_L': data['xi_over_L'],
    }

with open(save_dir / 'sweep_results.json', 'w') as f:
    json.dump(sweep_save, f, indent=2)

print(f'Saved to {save_dir.resolve()}')
if IN_COLAB:
    print('Results are persisted on your Google Drive automatically.')